# NM04. Mechanizmy atencji

Atencja jest łatwiejsza do zrozumienia, kiedy zaczniemy od prostego pytania: skąd model ma wiedzieć, na który fragment wejścia spojrzeć w danym momencie? Najpierw przypomnimy, jak współpracują enkoder i dekoder. Potem przejdziemy do atencji Bahdanau i Luonga, a następnie do macierzy Q/K/V (query/key/values), maskowania i atencji z wieloma głowami.

Kod działa na CPU. Potrzebne są Python 3, PyTorch, NumPy i Matplotlib.

**Oznaczenia:** 

`B` - liczba przykładów w paczce, 

`S` - liczba kluczy (zwykle długość wejścia), 

`T` - liczba zapytań, 

`D` - wymiar wektora. 

Używamy układu `batch-first`, np. `[B, S, D]`.

Zacznij od importu bibliotek.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
print("PyTorch:", torch.__version__)
print("Urządzenie: CPU (wystarczy do wszystkich przykładów)")

### 0. Zanim pojawi się atencja: enkoder i dekoder

Wyobraźmy sobie model, który dostaje sekwencję i ma na jej podstawie utworzyć inną. Może to być tłumaczenie zdania, zamiana liter na fonemy albo wygenerowanie krótkiej odpowiedzi. Długości wejścia i wyjścia nie muszą być takie same.

W klasycznym modelu **encoder–decoder** opartym na sieciach rekurencyjnych pracują dwie części:

- **Enkoder** czyta elementy wejścia po kolei: $x_1,x_2,\ldots,x_S$. Po każdym kroku ma nowy stan ukryty $h_j$. W prostej wersji ostatni stan $h_S$ jest podsumowaniem całego wejścia.
- **Dekoder** dostaje to podsumowanie jako stan początkowy i wytwarza sekwencję wyjściową: $y_1,y_2,\ldots,y_T$. W kolejnym kroku korzysta ze swojego stanu oraz poprzedniego symbolu wyjściowego. Generowanie zaczyna od specjalnego symbolu startu, a może kończyć symbolem końca sekwencji.

Można to zapisać tak:

```text
        x₁, x₂, ..., xₛ  →  ENKODER  →  h₁, h₂, ..., hₛ
                                                 │
                                        stan końcowy hₛ
                                                 │
                                                 ↓
                                      DEKODER: START → y₁ → y₂ → ...
```

Podczas uczenia często podajemy dekoderowi prawidłowy poprzedni symbol zamiast jego własnej predykcji. Nazywamy to *teacher forcing*. Podczas generowania model zwykle musi opierać się na symbolach, które sam wcześniej wygenerował.

**Przykład:** Jak przepływają dane między enkoderem a dekoderem?

Mamy trzy wektory wejściowe o wymiarze $3$ i chcemy wykonać dwa kroki dekodera. Obie sieci GRU mają stan ukryty o wymiarze $4$.

Enkoder zwraca stan dla każdej pozycji (`[B,S,4]`) oraz swój stan końcowy (`[1,B,4]`, bo używamy jednej warstwy GRU). Ten końcowy stan przekazujemy do dekodera. Następnie dekoder zwraca dwa własne stany, z których prosta warstwa liniowa tworzy oceny trzech możliwych symboli wyjściowych.

W przykładzie podajemy na wejściu dekodera dwa gotowe wektory: symbol startu i poprzedni prawidłowy symbol. To demonstracja *teacher forcing*, nie samodzielnego generowania. Wagi są losowe, więc liczby nie mają jeszcze znaczenia językowego.

In [ ]:
with torch.random.fork_rng():
    torch.manual_seed(7)
    encoder_0 = nn.GRU(input_size=3, hidden_size=4, batch_first=True)
    decoder_0 = nn.GRU(input_size=3, hidden_size=4, batch_first=True)
    output_layer_0 = nn.Linear(4, 3)

    x_0 = torch.tensor(
        [[[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]]]
    )  # [B=1, S=3, D=3]
    decoder_inputs_0 = torch.tensor(
        [[[1.0, 0.0, 0.0], [0.0, 1.0, 0.0]]]
    )  # START, poprzedni symbol

    with torch.no_grad():
        encoder_states_0, final_state_0 = encoder_0(x_0)
        decoder_states_0, _ = decoder_0(decoder_inputs_0, final_state_0)
        logits_0 = output_layer_0(decoder_states_0)
        probabilities_0 = torch.softmax(logits_0, dim=-1)

print("Stany enkodera:", tuple(encoder_states_0.shape))
print("Stan przekazany dekoderowi:", tuple(final_state_0.shape))
print("Stany dekodera:", tuple(decoder_states_0.shape))
print("Rozkłady prawdopodobieństwa:", tuple(probabilities_0.shape))
assert encoder_states_0.shape == (1, 3, 4)
assert final_state_0.shape == (1, 1, 4)
assert decoder_states_0.shape == (1, 2, 4)
assert probabilities_0.shape == (1, 2, 3)
assert torch.allclose(encoder_states_0[:, -1, :], final_state_0[0])
assert torch.allclose(probabilities_0.sum(dim=-1), torch.ones(1, 2))

Enkoder przeczytał $3$ elementy, dekoder wykonał $2$ kroki. Ich długości nie muszą się zgadzać. Dekoder w tej wersji otrzymał na początku tylko jeden wektor podsumowujący wejście, chociaż enkoder obliczył trzy stany. Do obliczenia `logits` nie używaliśmy osobno wcześniejszych stanów encodera.

**Przykład**: Co może zginąć w jednym wektorze?

Rozważmy dwa wejścia: `A, B` oraz `B, A`. Dla przejrzystości przypiszmy `A` wektor `[2,0]`, a `B` wektor `[0,2]`. 

Załóżmy teraz bardzo prosty enkoder, który streszcza sekwencję średnią jej wektorów. Zobaczmy, czy dekoder, który widzi tylko tę średnią, potrafi odróżnić kolejność elementów.

In [ ]:
A_0 = torch.tensor([2.0, 0.0])
B_0 = torch.tensor([0.0, 2.0])
seq_AB_0 = torch.stack([A_0, B_0])
seq_BA_0 = torch.stack([B_0, A_0])

summary_AB_0 = seq_AB_0.mean(dim=0)
summary_BA_0 = seq_BA_0.mean(dim=0)
print("Podsumowanie A,B:", summary_AB_0.tolist())
print("Podsumowanie B,A:", summary_BA_0.tolist())
assert torch.equal(summary_AB_0, summary_BA_0)

# Jeśli możemy wrócić do poszczególnych pozycji, kolejność jest rozróżnialna.
first_position_0 = torch.tensor([1.0, 0.0])
first_AB_0 = first_position_0 @ seq_AB_0
first_BA_0 = first_position_0 @ seq_BA_0
print("Pierwsza pozycja A,B:", first_AB_0.tolist())
print("Pierwsza pozycja B,A:", first_BA_0.tolist())
assert not torch.equal(first_AB_0, first_BA_0)

Oba wejścia mają średnią `[1,1]`, więc dekoder otrzymujący wyłącznie taką średnią nie pozna z niej kolejności. Gdy ma dostęp do reprezentacji poszczególnych pozycji, może wydobyć informację o pierwszym elemencie.

Prawdziwy enkoder GRU nie liczy zwykłej średniej i może zakodować kolejność w swoim stanie końcowym. Pokazuje jednak, dlaczego przekazywanie całego wejścia przez jeden wektor może być ograniczeniem. 

W kolejnych częściach zobaczymy, jak mechanizm atencji uczy się wybierać odpowiednie pozycje w zależności od potrzeb dekodera.

### 1. Dlaczego enkoder–dekoder potrzebuje atencji?

W prostym modelu encoder–decoder opartym na RNN całe wejście zostaje podsumowane jednym wektorem. Im dłuższe wejście, tym trudniej zmieścić w nim informacje potrzebne do wygenerowania wszystkich kolejnych elementów wyjścia.

Atencja Bahdanau pozwala w każdym kroku dekodera obliczać **nowy wektor kontekstu**, korzystając z różnych wag dla stanów encodera, tj.

$$e_{tj}=v^\top\tanh(W_qs_{t-1}+W_kh_j+b),$$

$$\alpha_{tj}=\operatorname{softmax}_j(e_{tj}),\qquad
c_t=\sum_{j=1}^{S}\alpha_{tj}h_j.$$

Indeks `t` oznacza krok generowania wyjścia. Indeks `j` wskazuje pozycję wejścia. Liczba kroków wyjścia nie musi być równa liczbie pozycji wejścia.

Załóżmy jedną sekwencję o trzech pozycjach (`S=3`) i dwa kroki dekodera (`T=2`). Każdy stan encodera jest dwuwymiarowy. 

Macierz wag ma kształt `[1,2,3]`, bo na każdy krok dekodera przypada osobny rozkład wag po trzech pozycjach wejścia. Dwa uzyskane konteksty mają kształt `[1,2,2]`.

In [ ]:
demo_H = torch.tensor([[[1.0, 0.0], [0.0, 2.0], [3.0, 1.0]]])  # [B=1, S=3, D=2]
demo_scores = torch.tensor([[[2.0, 0.0, 0.0], [0.0, 1.0, 2.0]]])  # [B=1, T=2, S=3]
demo_weights = torch.softmax(demo_scores, dim=-1)
demo_contexts = demo_weights @ demo_H
print("Stany encodera:", tuple(demo_H.shape))
print("Wagi:", tuple(demo_weights.shape))
print("Konteksty:", tuple(demo_contexts.shape))
print("Wagi w kroku 1:", demo_weights[0, 0].round(decimals=3).tolist())
print("Wagi w kroku 2:", demo_weights[0, 1].round(decimals=3).tolist())
assert demo_contexts.shape == (1, 2, 2)

Teraz przyjmij `B=2`, `S=5`, `T=3`, `D_enc=8` i `D_dec=4`.

1. Jakie są kształty tensorów `H` (stany encodera), `s_{t-1}` (jedno zapytanie dekodera), `A` (wagi wszystkich kroków) i `c_t` (kontekst dla jednego kroku)?

   *Odpowiedź:* 

2. Dlaczego w kolejnych krokach tłumaczenia lub zamiany liter na fonemy przydają się różne konteksty? Spróbuj podać jeden konkretny przykład.

   *Odpowiedź:* 

3. Czy już w pierwszym kroku dekoder może korzystać ze wszystkich stanów encodera? A czy podczas autoregresyjnego generowania wolno mu podejrzeć przyszłe tokeny wyjściowe? 

   *Odpowiedź:* 

### 2. Od ocen dopasowania do wektora kontekstu

Atencja składa się z trzech kroków: najpierw liczymy **oceny dopasowania** (*scores*), potem zamieniamy je funkcją `softmax` na **wagi**, a na końcu obliczamy **sumę ważoną wartości**. Warto odróżniać te trzy rzeczy.

**Przykład:** 

Niech oceny wynoszą $[\ln 1,\ln 3]$, a wektory wartości to $[2,0]$ oraz $[0,4]$.

- Po zastosowaniu $\exp(x)$ względem współrzędnych wektora ocen otrzymujemy proporcje $1:3$.
- Po normalizacji wagi wynoszą $[1/4,3/4]$ i sumują się do $1$.
- Kontekst to $(1/4)[2,0]+(3/4)[0,4]=[0.5,3]$.

W kodzie można to zapisać wprost. Tutaj używamy gotowego `torch.softmax`, żeby skupić się na znaczeniu obliczenia, a nie na implementacji funkcji.

In [ ]:
demo_e = torch.log(torch.tensor([1.0, 3.0]))
demo_values = torch.tensor([[2.0, 0.0], [0.0, 4.0]])
demo_alpha = torch.softmax(demo_e, dim=0)
demo_context = demo_alpha @ demo_values
print("Wagi:", demo_alpha.tolist())
print("Kontekst:", demo_context.tolist())
assert torch.allclose(demo_alpha, torch.tensor([0.25, 0.75]))
assert torch.allclose(demo_context, torch.tensor([0.5, 3.0]))

Powtórz procedurę dla trzech wartości. Weźmy następujące oceny:

$$e=[\ln 1,\;\ln 2,\;\ln 4]$$

i wartości:

$$H=\begin{bmatrix}1&2\\2&0\\0&4\end{bmatrix}.$$

Najpierw policz $\alpha=\operatorname{softmax}(e)$ i $c=\alpha H$. Zapisz ułamki, zanim przejdziesz do liczb dziesiętnych.

Następnie uzupełnij dwie funkcje: `stable_softmax` (bez wywoływania `torch.softmax`) oraz `context_from_scores`, która działa również dla całej paczki. W stabilnym softmaksie przed wywołaniem `exp` odejmujemy największą ocenę z danego wiersza — to nie zmienia rozkładu, a chroni przed przepełnieniem.

Uwaga: gdy wszystkie oceny w wierszu wynoszą `-inf`, zwykły softmax nie jest zdefiniowany i może zwrócić `NaN`. W tym notebooku zakładamy, że istnieje co najmniej jedna dozwolona pozycja w każdym wierszu.

**Miejsce na ewentualne obliczenia:**

...

In [ ]:
def stable_softmax(x: torch.Tensor, dim: int = -1) -> torch.Tensor:
    """Softmax bez przepełnienia exp dla bardzo dużych wartości."""
    # TODO: odejmij maksimum wzdłuż `dim`, potem exp i normalizacja.
    raise NotImplementedError("softmax i kontekst: stable_softmax")


def context_from_scores(scores: torch.Tensor, values: torch.Tensor):
    """scores [B,S], values [B,S,D] -> (alpha [B,S], context [B,D])."""
    # TODO: użyj stable_softmax, a następnie sumy ważonej stanów.
    raise NotImplementedError("softmax i kontekst: context_from_scores")

In [ ]:
e = torch.log(torch.tensor([[1.0, 2.0, 4.0]]))
H = torch.tensor([[[1.0, 2.0], [2.0, 0.0], [0.0, 4.0]]])
a, c = context_from_scores(e, H)
assert a.shape == (1, 3) and c.shape == (1, 2)
assert torch.allclose(a[0], torch.tensor([1 / 7, 2 / 7, 4 / 7]), atol=1e-6)
assert torch.allclose(c[0], torch.tensor([5 / 7, 18 / 7]), atol=1e-6)

# Stabilność, niezmienniczość na dodanie stałej i normalizacja po wskazanej osi.
large = torch.tensor([[1000.0, 1001.0], [-1000.0, -999.0]])
assert torch.isfinite(stable_softmax(large, dim=-1)).all()
assert torch.allclose(stable_softmax(large), torch.softmax(large, dim=-1), atol=1e-6)
assert torch.allclose(
    stable_softmax(e.double() + 1000), stable_softmax(e.double()), atol=1e-12
)
assert torch.allclose(stable_softmax(large.T, dim=0).sum(0), torch.ones(2), atol=1e-6)

# Dwa przykłady z różnymi wartościami muszą dać osobne konteksty.
a2, c2 = context_from_scores(torch.zeros(2, 3), torch.stack([H[0], 2 * H[0]]))
assert a2.shape == (2, 3) and c2.shape == (2, 2)
assert torch.allclose(c2[1], 2 * c2[0], atol=1e-6)
print("✓ softmax i kontekst: softmax, stabilność i wektory kontekstu")

**Pytania.**

- Dlaczego dodanie tej samej stałej (nawet bardzo dużej) do wszystkich ocen nie zmienia wag softmaxu?
- Dlaczego normalizujemy po pozycjach wejścia, a nie po współrzędnych pojedynczego wektora wartości?

**Odpowiedź**: ...

### 3. Atencja Bahdanau

W atencji addytywnej Bahdanau ocena dopasowania zależy zarówno od stanu dekodera, jak i od danego stanu encodera:

$$e_{tj}=v^\top\tanh(W_qs_{t-1}+W_kh_j+b).$$

Żeby policzyć oceny dla wszystkich pozycji jednocześnie, wykorzystujemy broadcasting (w Pythonie). Projekcja zapytania ma kształt `[B, d_attn]`, a projekcje stanów encodera `[B, S, d_attn]`. Dodanie osi długości `1` do zapytania pozwala zsumować je z każdą pozycją wejścia.

In [ ]:
# Przykład: jeden wektor zapytania, trzy pozycje wejścia.
demo_q_proj = torch.tensor([[1.0, 2.0]])  # [B=1, d_attn=2]
demo_h_proj = torch.tensor(
    [[[0.0, 1.0], [2.0, -1.0], [10.0, 10.0]]]
)  # [B=1, S=3, d_attn=2]
demo_combined = torch.tanh(demo_q_proj.unsqueeze(1) + demo_h_proj)
print("Po broadcastingu:", tuple(demo_combined.shape))
assert demo_combined.shape == (1, 3, 2)

# Osobno pokazujemy maskę ocen. Ostatnia pozycja to padding.
demo_scores_mask = torch.tensor([[0.0, 1.0, 40.0]])
demo_allowed = torch.tensor([[True, True, False]])
demo_masked = demo_scores_mask.masked_fill(~demo_allowed, float("-inf"))
demo_masked_weights = torch.softmax(demo_masked, dim=-1)
print("Wagi po zamaskowaniu:", demo_masked_weights.tolist())
assert demo_masked_weights[0, 2].item() == 0.0
assert torch.allclose(demo_masked_weights.sum(-1), torch.ones(1))

Połącz powyższe elementy w klasie `BahdanauAttention`. Parametry projekcji są już zdefiniowane. Uzupełnij wyłącznie metodę `forward`, bez pętli po przykładach ani po pozycjach wejścia.

- `query` to stan dekodera, `keys` to stany encodera. W tej wersji stany encodera są jednocześnie **kluczami** (do wyznaczania wag) i **wartościami** (values; do utworzenia kontekstu).
- `mask=True` oznacza, że pozycja jest prawdziwa; `False` oznacza padding. Zabronione oceny ustawiamy na `-inf` przed softmaxem.
- W `self.W_k` włączony jest `bias`, więc nie dodawaj osobno $b$.
- W każdej sekwencji pozostaje przynajmniej jedna dozwolona pozycja.

In [ ]:
class BahdanauAttention(nn.Module):
    def __init__(self, d_query: int, d_key: int, d_attn: int):
        super().__init__()
        self.W_q = nn.Linear(d_query, d_attn, bias=False)
        self.W_k = nn.Linear(d_key, d_attn, bias=True)
        self.v = nn.Linear(d_attn, 1, bias=False)

    def forward(self, query, keys, mask):
        """
        query: [B,d_query]          poprzedni stan dekodera
        keys:  [B,S,d_key]          stany encodera
        mask:  [B,S] bool           True = prawdziwy element
        zwraca: context [B,d_key], alpha [B,S]
        """
        # TODO 1: projekcje query i keys oraz ich suma przez broadcasting.
        # TODO 2: tanh, projekcja v -> scores [B,S].
        # TODO 3: maskowanie przed softmax -> alpha.
        # TODO 4: ważona suma oryginalnych `keys` -> context.
        raise NotImplementedError("atencja Bahdanau: BahdanauAttention.forward")

In [ ]:
# Test dla znanego wyniku liczbowego, łącznie z tanh i maskowaniem.
att_known = BahdanauAttention(d_query=1, d_key=1, d_attn=1)
with torch.no_grad():
    att_known.W_q.weight.fill_(1.0)
    att_known.W_k.weight.fill_(1.0)
    att_known.W_k.bias.zero_()
    att_known.v.weight.fill_(1.0)
q_known = torch.tensor([[0.0]])
enc_known = torch.tensor([[[0.0], [1.0], [100.0]]])
valid_known = torch.tensor([[True, True, False]])
ctx_known, a_known = att_known(q_known, enc_known, valid_known)
expected_known = torch.softmax(torch.tensor([0.0, math.tanh(1.0)]), dim=0)
assert torch.allclose(a_known[0, :2], expected_known, atol=1e-6)
assert a_known[0, 2].item() == 0.0
assert torch.allclose(ctx_known.squeeze(), expected_known[1], atol=1e-6)

# Test z dwoma przykładami, wymiary i przepływami gradientu przez wszystkie parametry.
torch.manual_seed(SEED)
att = BahdanauAttention(d_query=3, d_key=4, d_attn=5)
q = torch.randn(2, 3, requires_grad=True)
enc = torch.randn(2, 4, 4, requires_grad=True)
valid = torch.tensor([[True, True, False, False], [True, True, True, False]])
ctx, alpha = att(q, enc, valid)
assert ctx.shape == (2, 4) and alpha.shape == (2, 4)
assert torch.isfinite(ctx).all() and torch.isfinite(alpha).all()
assert torch.all(alpha[~valid] == 0)
assert torch.allclose(alpha.sum(-1), torch.ones(2), atol=1e-6)
assert torch.allclose(ctx, (alpha.unsqueeze(-1) * enc).sum(1), atol=1e-6)
ctx.square().sum().backward()
for name, parameter in att.named_parameters():
    assert parameter.grad is not None, f"Brak gradientu: {name}"
    assert torch.isfinite(parameter.grad).all(), f"Nieskończony gradient: {name}"
    assert parameter.grad.abs().sum().item() > 1e-9, f"Zerowy gradient: {name}"
for name, tensor in [("query", q), ("enc", enc)]:
    assert tensor.grad is not None and torch.isfinite(tensor.grad).all(), name
    assert tensor.grad.abs().sum().item() > 1e-9, name
assert torch.all(enc.grad[~valid] == 0), "Padding nie powinien wpływać na stratę"
print("Atencja Bahdanau: wynik liczbowy, maska i gradienty")

**Pytania.**

- Dlaczego podstawienie `0` zamiast `-inf` nie wystarcza do ukrycia pozycji paddingu?
- Dlaczego kontekst tworzymy z wektorów wartości, a nie z samych ocen? Czy klucze i wartości zawsze muszą być tym samym tensorem?

**Odpowiedź**:

### Na marginesie: jak czytać mapę atencji przy zamianie liter na fonemy?

W zadaniu G2P (*grapheme-to-phoneme*) model widzi litery, a generuje fonemy. Wyobraźmy sobie słowo **phone** zapisane jako `p h o n e` i wynik `F OW N`.

Wiersz odpowiada fonemowi, kolumna odpowiada literze. Każdy wiersz sumuje się do $1$, bo są to wagi atencji dla jednego kroku dekodera. Czasem sprawdza się także *coverage*, czyli sumę wag otrzymanych przez daną literę we wszystkich krokach:

$$\mathrm{coverage}_j=\sum_t\alpha_{tj}.$$

**Przykład**. Dla litery `n` z poniższej macierzy otrzymujemy $0.03+0.04+0.85=0.92$. To suma po kolumnie, więc nie musi być równa $1$. Uruchom kod, żeby obejrzeć całą mapę.

In [ ]:
letters = ["p", "h", "o", "n", "e"]
phonemes = ["F", "OW", "N"]
A_g2p = torch.tensor(
    [
        [0.45, 0.45, 0.05, 0.03, 0.02],
        [0.02, 0.02, 0.72, 0.04, 0.20],
        [0.02, 0.03, 0.05, 0.85, 0.05],
    ]
)
coverage = A_g2p.sum(dim=0)
print("Coverage:", {k: round(v, 2) for k, v in zip(letters, coverage.tolist())})

fig, ax = plt.subplots(figsize=(6.5, 3.1), layout="constrained")
im = ax.imshow(A_g2p, cmap="viridis", vmin=0, vmax=1)
ax.set(
    xticks=range(5),
    xticklabels=letters,
    yticks=range(3),
    yticklabels=phonemes,
    xlabel="Litera wejścia",
    ylabel="Fonem wyjścia",
    title="Przykładowa mapa G2P",
)
fig.colorbar(im, ax=ax, shrink=0.8)
plt.show()

Spójrz na wynik: które litery mają największy *coverage* i czy duża waga przy `p` oraz `h` dla fonemu `F` ma sens intuicyjny?

Uwaga: wysoka waga atencji nie dowodzi, że dany znak był przyczyną predykcji. Sama mapa nie zastępuje analizy zachowania modelu po zmianie wejścia.

**Odpowiedź**: 

### 4. Inny sposób liczenia ocen: atencja Luonga

Bahdanau wykorzystuje projekcje, dodawanie i `tanh`. Luong zaproponował także prostszą ocenę dopasowania, w wariancie *general* jest to iloczyn skalarny z uczoną macierzą:

$$e_{tj}=s_{t-1}^{\top}Wh_j.$$

W oryginalnej pracy Luonga występuje bieżący stan dekodera $h_t$. Tutaj korzystamy z $s_{t-1}$, żeby zachować ten sam interfejs co wcześniej. Nie dodajemy też dzielenia przez $\sqrt{d_k}$, to pojawia się dopiero w pełnoprawnych Transformerach.

**Przykład.** Niech $q=[1,1]$, $k=[2,3]$ i $W=\operatorname{diag}(2,1)$. Projekcja klucza to $Wk=[4,3]$, a ocena dopasowania wynosi $q^\top Wk=7$. To tylko jedna liczba; do uzyskania wag dla całej sekwencji nadal potrzebny będzie softmax.

In [ ]:
demo_q_luong = torch.tensor([1.0, 1.0])
demo_k_luong = torch.tensor([2.0, 3.0])
demo_W_luong = torch.diag(torch.tensor([2.0, 1.0]))
demo_projected = demo_W_luong @ demo_k_luong
demo_score = demo_q_luong @ demo_projected
print("Wk =", demo_projected.tolist(), "score =", demo_score.item())
assert demo_score.item() == 7.0

Uzupełnij `LuongGeneralAttention.forward`. Macierz `W` znajduje się już w warstwie `self.W`. Dla wszystkich pozycji jednocześnie oblicz oceny dopasowania, zastosuj maskę i softmax, a potem zsumuj **oryginalne** stany encodera przy użyciu wyznaczonych wag.

Porównaj z Bahdanau: w którym miejscu Luong rezygnuje z nieliniowości i wektora `v`?

In [ ]:
class LuongGeneralAttention(nn.Module):
    def __init__(self, d_query: int, d_key: int):
        super().__init__()
        self.W = nn.Linear(d_key, d_query, bias=False)

    def forward(self, query, keys, mask):
        """query [B,d_query], keys [B,S,d_key], mask [B,S] bool.
        Zwróć: context [B,d_key], alpha [B,S].
        """
        # TODO: projected_keys = self.W(keys); oblicz dot-product dla każdej pozycji,
        # następnie maskowanie, softmax i sumę ważoną.
        raise NotImplementedError("atencja Luonga: LuongGeneralAttention.forward")

In [ ]:
dot = LuongGeneralAttention(d_query=2, d_key=2)
with torch.no_grad():
    dot.W.weight.copy_(torch.eye(2))  # test na znanej macierzy W
q_toy = torch.tensor([[1.0, 0.0]])
k_toy = torch.tensor([[[1.0, 0.0], [0.0, 1.0], [100.0, 100.0]]])
m_toy = torch.tensor([[True, True, False]])
c_toy, a_toy = dot(q_toy, k_toy, m_toy)
expected = torch.softmax(torch.tensor([1.0, 0.0]), dim=0)
assert a_toy.shape == (1, 3) and c_toy.shape == (1, 2)
assert torch.allclose(a_toy[0, :2], expected, atol=1e-6)
assert a_toy[0, 2].item() == 0.0
assert torch.allclose(c_toy[0], expected, atol=1e-6)
print("Atencja Luonga: bilinear dot-product i ignorowanie paddingu")

Do porównania tych dwóch metod nie wystarczy jeden przebieg przez losowo zainicjalizowane warstwy. Jak zaprojektowałbyś prosty, uczciwy eksperyment, żeby sprawdzić wpływ rodzaju atencji na jakość modelu?

*Odpowiedź:* 

### 5. Scaled dot-product attention

Do tej pory stany encodera mogły jednocześnie służyć do porównywania i agregowania informacji. Ogólny zapis rozdziela te role:

- **Query (Q)** — czego szuka dana pozycja?
- **Key (K)** — z czym porównujemy zapytanie, aby policzyć oceny?
- **Value (V)** — jakie informacje połączymy w wyniku?

W self-attention najczęściej zaczynamy od tego samego wejścia $X$, lecz uczymy osobne projekcje:

$$Q=XW_Q,\qquad K=XW_K,\qquad V=XW_V.$$

Następnie liczymy:

$$\operatorname{Attention}(Q,K,V)=\operatorname{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V.$$

W cross-attention zapytania mogą pochodzić z dekodera, a klucze i wartości z encodera. Dlatego liczba zapytań `T` może być inna niż liczba kluczy `S`. Również wymiary wektorów K i V nie muszą być jednakowe.

**Przykład** 

Niech zapytanie wynosi $q=[1,0]$, klucze $k_1=[1,0]$, $k_2=[0,1]$, a wartości $v_1=[2,0]$, $v_2=[0,8]$. Oceny przed skalowaniem to `[1,0]`. Dla $d_k=2$ dzielimy je przez $\sqrt{2}$, stosujemy softmax i mieszamy wartości, a nie klucze.

In [ ]:
# Przykład rozwiązany: Q i K wyznaczają wagi, V tworzy wynik.
demo_Q = torch.tensor([[1.0, 0.0]])
demo_K = torch.tensor([[1.0, 0.0], [0.0, 1.0]])
demo_V = torch.tensor([[2.0, 0.0], [0.0, 8.0]])
demo_raw = demo_Q @ demo_K.T
demo_scaled = demo_raw / math.sqrt(2)
demo_attn = torch.softmax(demo_scaled, dim=-1)
demo_result = demo_attn @ demo_V
print("Oceny:", demo_raw.tolist())
print("Po skalowaniu:", demo_scaled.tolist())
print("Wagi:", demo_attn.tolist())
print("Wynik:", demo_result.tolist())
assert demo_attn.shape == (1, 2) and demo_result.shape == (1, 2)

Policz teraz to samo dla $q_1=[2,0]$,

$$K=\begin{bmatrix}2&0\\0&2\end{bmatrix},\qquad
V=\begin{bmatrix}1&10\\10&1\end{bmatrix}.$$

Zapisz dwie oceny $q_1\cdot k_j$, ich wartości po podzieleniu przez $\sqrt{2}$, wagi softmax i wynik $o_1$. Możesz zostawić wagi w postaci zawierającej funkcję wykładniczą.

Potem uzupełnij `scaled_dot_product_attention`, która przyjmuje już przygotowane tensory Q, K i V. Atencja jest zaimplementowana w PyTorchu jako `torch.nn.functional.scaled_dot_product_attention`, ale nie używamy jej do rozwiązania tego zadania.

| Tensor | Wymiary |
|---|---|
| `Q` | `[B, T, d_k]` |
| `K` | `[B, S, d_k]` |
| `V` | `[B, S, d_v]` |
| `weights` | `[B, T, S]` |
| `output` | `[B, T, d_v]` |

W naszej funkcji `key_mask=True` oznacza dozwolony klucz. Dla uproszczenia, jeśli `causal=True`, wymagamy `T == S`; przy innych długościach funkcja powinna zgłaszać `ValueError`. 

Dekodowanie przyrostowe z KV cache wymaga bardziej ogólnej maski, ale nie implementujemy go tutaj. Każdy wiersz musi mieć przynajmniej jeden dozwolony klucz.

Miejsce na ewentualne obliczenia:

...

In [ ]:
def scaled_dot_product_attention(Q, K, V, key_mask=None, causal=False):
    """
    Q: [B,T,d_k], K: [B,S,d_k], V: [B,S,d_v]
    key_mask: [B,S] bool; True = dozwolony klucz (opcjonalnie)
    causal: jeżeli True, zapytanie t widzi tylko klucze j <= t (wymagamy T == S)
    zwraca: output [B,T,d_v], weights [B,T,S]

    Założenie: każdy wiersz, po uwzględnieniu masek, ma >= 1 dozwolony klucz.
    """
    # TODO 1: gdy causal=True i T!=S, zgłoś ValueError z czytelnym komunikatem.
    # TODO 2: scores = (Q @ K.transpose(-2, -1)) / math.sqrt(d_k)
    # TODO 3: key_mask -> [B,1,S]; zabronione oceny ustaw na -inf.
    # TODO 4: causal mask: j > t (np. torch.triu(..., diagonal=1) na urządzeniu Q).
    # TODO 5: softmax wzdłuż S; output = weights @ V. Zwróć (output, weights).
    raise NotImplementedError("Q/K/V: scaled_dot_product_attention")

In [ ]:
# Wynik z ręcznego rachunku.
Q0 = torch.tensor([[[2.0, 0.0], [0.0, 2.0]]])
K0 = torch.tensor([[[2.0, 0.0], [0.0, 2.0]]])
V0 = torch.tensor([[[1.0, 10.0], [10.0, 1.0]]])
O0, A0 = scaled_dot_product_attention(Q0, K0, V0)
expected_row = torch.softmax(torch.tensor([4 / math.sqrt(2), 0.0]), dim=0)
assert O0.shape == (1, 2, 2) and A0.shape == (1, 2, 2)
assert torch.allclose(A0[0, 0], expected_row, atol=1e-6)
assert torch.allclose(O0[0, 0], expected_row @ V0[0], atol=1e-6)

# Cross-attention: różne T/S, różne d_k/d_v.
torch.manual_seed(104)
Qcross, Kcross, Vcross = (
    torch.randn(3, 2, 4),
    torch.randn(3, 5, 4),
    torch.randn(3, 5, 6),
)
Ocross, Across = scaled_dot_product_attention(Qcross, Kcross, Vcross)
assert Ocross.shape == (3, 2, 6) and Across.shape == (3, 2, 5)
assert torch.allclose(Across.sum(-1), torch.ones(3, 2), atol=1e-6)
expected_cross = torch.softmax(Qcross @ Kcross.transpose(-2, -1) / math.sqrt(4), dim=-1)
assert torch.allclose(Across, expected_cross, atol=1e-6)
assert torch.allclose(Ocross, expected_cross @ Vcross, atol=1e-6)

# Key mask: wynik musi ignorować bardzo duży, niedozwolony klucz.
k_mask = torch.tensor([[True, True, False]])
q_mask = torch.tensor([[[1.0, 0.0]]])
keys_mask = torch.tensor([[[0.0, 0.0], [1.0, 0.0], [1000.0, 0.0]]])
vals_mask = torch.tensor([[[1.0], [3.0], [9999.0]]])
o_mask, a_mask = scaled_dot_product_attention(
    q_mask, keys_mask, vals_mask, key_mask=k_mask
)
expected_mask = torch.softmax(torch.tensor([0.0, 1 / math.sqrt(2)]), dim=0)
assert torch.allclose(a_mask[0, 0, :2], expected_mask, atol=1e-6)
assert a_mask[0, 0, 2].item() == 0.0
assert torch.allclose(o_mask.squeeze(), expected_mask @ vals_mask[0, :2, 0], atol=1e-6)

# Causal i dwie maski: przyszłe klucze nie mogą wpływać na wcześniejsze query.
unit = torch.eye(3).unsqueeze(0)
_, Acausal = scaled_dot_product_attention(unit, unit, unit, causal=True)
upper3 = torch.triu(torch.ones(3, 3, dtype=torch.bool), diagonal=1)
assert torch.all(Acausal[0, upper3] == 0)
assert torch.allclose(Acausal.sum(-1), torch.ones(1, 3), atol=1e-6)
_, Aboth = scaled_dot_product_attention(
    unit, unit, unit, key_mask=torch.tensor([[True, True, False]]), causal=True
)
assert torch.all(Aboth[0, upper3] == 0) and torch.all(Aboth[..., 2] == 0)
try:
    scaled_dot_product_attention(Qcross, Kcross, Vcross, causal=True)
except ValueError:
    pass
else:
    raise AssertionError("Dla causal=True i T!=S oczekiwano ValueError")
print("Q/K/V: wyniki liczbowe, cross-attention, key mask i causal mask")

**Kolejny krok.** 

Poniżej znajduje się gotowa klasa `TinySelfAttention`. Pokazuje, jak trzy warstwy liniowe tworzą Q, K i V z jednego tensora `X`. Uruchom ją po uzupełnieniu `scaled_dot_product_attention`, obie części korzystają z tej samej funkcji.

Zwróć uwagę, że tensor wejścia ma wymiar `d_model=8`, ale klucze mają wymiar `d_k=4`, a wartości `d_v=5`. Przypominamy, że nie trzeba narzucać im tego samego wymiaru.

In [ ]:
class TinySelfAttention(nn.Module):
    def __init__(self, d_model=8, d_k=4, d_v=5):
        super().__init__()
        self.W_q = nn.Linear(d_model, d_k, bias=False)
        self.W_k = nn.Linear(d_model, d_k, bias=False)
        self.W_v = nn.Linear(d_model, d_v, bias=False)

    def forward(self, X):
        """X [B,S,d_model] -> (out [B,S,d_v], alpha [B,S,S])."""
        Q, K, V = self.W_q(X), self.W_k(X), self.W_v(X)
        return scaled_dot_product_attention(Q, K, V)


torch.manual_seed(4)
tiny = TinySelfAttention(d_model=8, d_k=4, d_v=5)
X = torch.randn(2, 3, 8, requires_grad=True)
Y, W = tiny(X)
assert Y.shape == (2, 3, 5) and W.shape == (2, 3, 3)
assert torch.isfinite(Y).all() and torch.isfinite(W).all()
assert torch.allclose(W.sum(-1), torch.ones(2, 3), atol=1e-6)
Y.square().sum().backward()
for name, param in tiny.named_parameters():
    assert param.grad is not None, f"Brak gradientu: {name}"
    assert torch.isfinite(param.grad).all(), f"Nieskończony gradient: {name}"
    assert param.grad.abs().sum().item() > 1e-9, f"Zerowy gradient: {name}"
assert X.grad is not None and torch.isfinite(X.grad).all()
assert X.grad.abs().sum().item() > 1e-9
print("✓ Q/K/V: trzy projekcje Q/K/V i autograd")

**Pytania**:

a) Czym różni się self-attention od cross-attention? Skąd pochodzą w nich Q, K i V?

b) Dlaczego uczymy trzy projekcje `W_q`, `W_k` i `W_v`, zamiast zawsze przyjmować `Q=K=V=X`?

c) Po co dzielimy oceny przez $\sqrt{d_k}$?

**Odpowiedzi:**

a)

b)

c)

### 6. Maskowanie: czego model nie powinien widzieć?

Są dwa częste powody ukrywania pozycji:

- **Padding mask** pomija sztuczne pozycje dopisane, żeby sekwencje w paczce miały równą długość. Nasze `key_mask=False` oznacza „nie używaj tego klucza”. To nie zeruje automatycznie wyjścia zapytania z pozycji paddingu.
- **Causal mask** nie pozwala zapytaniu w kroku `t` zobaczyć kluczy z pozycji `j > t`. Jest potrzebna w autoregresywnej self-attention.

W klasycznej atencji encodera–dekodera wejście jest już znane w całości, więc krok wyjściowy może korzystać nawet z ostatniego stanu encodera. Nie wolno natomiast podczas generowania podglądać przyszłych wyjść.

**Przykład:** maskowanie przed softmaxem. Jeśli padding ma bardzo dużą ocenę, softmax przypisze mu niemal całą wagę. Samo wyzerowanie takiej wagi po softmaksie zostawi pozostałe wagi bez odpowiedniej normalizacji. Najpierw więc ustawiamy niedozwolone oceny na `-inf`, a dopiero potem wywołujemy softmax.

In [ ]:
demo_s = torch.tensor([[2.0, 1.0, 30.0]])
demo_valid = torch.tensor([[True, True, False]])
demo_wrong = torch.softmax(demo_s, dim=-1).masked_fill(~demo_valid, 0.0)
demo_right = torch.softmax(demo_s.masked_fill(~demo_valid, float("-inf")), dim=-1)
print("Po softmaxie i wyzerowaniu (błędnie):", demo_wrong.tolist())
print("Po maskowaniu ocen (poprawnie):", demo_right.tolist())
print("Sumy wag:", demo_wrong.sum(-1).item(), demo_right.sum(-1).item())
assert demo_wrong.sum().item() < 0.01
assert torch.allclose(demo_right.sum(-1), torch.ones(1))

Kod poniżej korzysta z funkcji z rozdziału 5 i pokazuje cztery mapy dla tych samych zapytań, kluczy i wartości: bez maski, tylko z paddingiem, tylko z maską przyczynową i z obiema maskami. Wszystkie mapy mają tę samą skalę kolorów.

Po uruchomieniu porównaj kolumny (klucze) i wiersze (zapytania). To rozróżnienie często rozwiązuje nieporozumienia związane z maskami.

In [ ]:
torch.manual_seed(8)
Qm, Km, Vm = torch.randn(2, 4, 4), torch.randn(2, 4, 4), torch.randn(2, 4, 3)
valid_m = torch.tensor([[True, True, True, False], [True, True, False, False]])
_, A_free = scaled_dot_product_attention(Qm, Km, Vm)
_, A_pad = scaled_dot_product_attention(Qm, Km, Vm, key_mask=valid_m)
_, A_causal = scaled_dot_product_attention(Qm, Km, Vm, causal=True)
_, A_both = scaled_dot_product_attention(Qm, Km, Vm, key_mask=valid_m, causal=True)

for weights in (A_free, A_pad, A_causal, A_both):
    assert torch.isfinite(weights).all()
    assert torch.allclose(weights.sum(-1), torch.ones(2, 4), atol=1e-6)
blocked = ~valid_m[:, None, :].expand(-1, 4, -1)
assert torch.all(A_pad.masked_select(blocked) == 0)
upper = torch.triu(torch.ones(4, 4, dtype=torch.bool), diagonal=1)
assert torch.all(A_causal[:, upper] == 0)
assert torch.all(A_both[:, upper] == 0)
assert torch.all(A_both.masked_select(blocked) == 0)
# Maska kluczy nie wyzerowuje wierszy odpowiadających paddingowym query.
assert torch.allclose(A_pad[1, 2:].sum(-1), torch.ones(2), atol=1e-6)

fig, axes = plt.subplots(2, 2, figsize=(9, 7), layout="constrained")
# Przykład 1 (indeks 1) ma dwa prawdziwe klucze i dwa paddingowe.
for ax, weights, title in zip(
    axes.flat,
    [A_free[1], A_pad[1], A_causal[1], A_both[1]],
    ["Bez maski", "Tylko padding", "Tylko causal", "Padding + causal"],
):
    im = ax.imshow(weights.detach().cpu().numpy(), vmin=0, vmax=1, cmap="viridis")
    ax.set(title=title, xlabel="Pozycja klucza j (0–3)", ylabel="Pozycja query t (0–3)")
    ax.set_xticks(range(4))
    ax.set_yticks(range(4))
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.72, label="Waga atencji")
plt.show()
print("✓ maskowanie: cztery warianty, poprawna normalizacja i maski")

**Pytania:**

a) Co dokładnie psuje się w wyniku, gdy wyzerujemy niepożądane wagi dopiero po softmaksie i ich ponownie nie unormujemy?

b) Dlaczego atencja Bahdanau może patrzeć na ostatnią pozycję wejścia od pierwszego kroku dekodera, a autoregresywna self-attention nie może spojrzeć na przyszłą pozycję wyjścia?

c) Na mapie „Padding + causal” wskaż kolumny wyzerowane przez padding. Dlaczego wiersze należące do paddingowych zapytań nadal mogą zawierać niezerowe wagi?

**Odpowiedzi:**

a)

b)

c)

### 7. Po co kilka głów atencji? (multi-head attention)

Jedna głowa oblicza jedną macierz wag i jedną mieszankę wartości. W multi-head attention wykonujemy kilka takich operacji z osobnymi uczonymi projekcjami, łączymy wyniki i stosujemy projekcję wyjściową:

$$\operatorname{head}_i=\operatorname{Attention}(QW_i^Q,KW_i^K,VW_i^V),$$

$$\operatorname{MultiHead}=\operatorname{Concat}(\operatorname{head}_1,\ldots,\operatorname{head}_h)W^O.$$

Dla `d_model=8` i `n_heads=2` każda głowa pracuje na wektorach wymiaru `4`.

**Przykład** .

W poniższym kodzie tylko dzielimy tensor na części, żeby zobaczyć wymiary. To nie jest jeszcze mechanizm multi-head attention: prawdziwa warstwa najpierw wyznacza uczone projekcje Q, K i V.

In [ ]:
# Przykład rozwiązany: obejrzenie wymiaru głowy, jeszcze bez obliczania atencji.
demo_X_heads = torch.arange(1 * 3 * 8, dtype=torch.float32).reshape(1, 3, 8)
demo_split = demo_X_heads.reshape(1, 3, 2, 4).transpose(1, 2)
print("Wejście [B,S,d_model]:", tuple(demo_X_heads.shape))
print("Podział [B,n_heads,S,d_head]:", tuple(demo_split.shape))
assert demo_split.shape == (1, 2, 3, 4)

W PyTorchu mamy funkcje: `nn.MultiheadAttention`, `batch_first=True` oraz `average_attn_weights=False`, aby dostać osobną mapę wag dla każdej głowy. Dla porównania obejrzymy dwie sekwencje: bez paddingu i z paddingiem.

W naszej funkcji `key_mask=True` znaczy *zachowaj klucz*, ale w `nn.MultiheadAttention(key_padding_mask)` wartość `True` oznacza *ignoruj klucz*. Maska kluczy zeruje wybrane kolumny macierzy atencji, nie wiersze zapytań. Nawet jeśli paddingowe wektory wejściowe są zerowe, wyjścia na tych pozycjach mogą pozostać niezerowe. Podczas uczenia zwykle ignorujemy je przy liczeniu straty lub obsługujemy osobną maską.

Głowy w tym przykładzie są zainicjalizowane losowo. Różne mapy nie są dowodem, że jedna głowa rozumie składnię, a druga semantykę.

In [ ]:
torch.manual_seed(11)
x_mha = torch.randn(2, 5, 8)
x_mha[1, 3:] = 0.0  # także wartości wejściowe paddingu są zerowe
ignore_mha = torch.tensor(
    [[False, False, False, False, False], [False, False, False, True, True]]
)
mha = nn.MultiheadAttention(embed_dim=8, num_heads=2, batch_first=True, dropout=0.0)
mha.eval()  # obserwujemy same mapy bez losowych efektów dropout
with torch.no_grad():
    mha_output, head_weights = mha(
        x_mha,
        x_mha,
        x_mha,
        key_padding_mask=ignore_mha,
        need_weights=True,
        average_attn_weights=False,
    )

assert mha_output.shape == (2, 5, 8)
assert head_weights.shape == (2, 2, 5, 5)  # [batch, head, query, key]
assert torch.isfinite(head_weights).all()
assert torch.allclose(head_weights.sum(-1), torch.ones(2, 2, 5), atol=1e-6)
assert torch.all(head_weights[1, :, :, 3:] == 0)  # klucze paddingowe
# Paddingowe zapytania (wiersze 3,4) nadal mają znormalizowane wagi.
assert torch.allclose(head_weights[1, :, 3:, :3].sum(-1), torch.ones(2, 2), atol=1e-6)
assert mha_output[1, 3:].abs().sum().item() > 0

fig, axes = plt.subplots(2, 2, figsize=(9, 7), layout="constrained")
for b in range(2):
    for h in range(2):
        ax = axes[b, h]
        im = ax.imshow(head_weights[b, h].cpu().numpy(), vmin=0, vmax=1, cmap="viridis")
        ax.set(
            title=f"Przykład {b + 1}: głowa {h + 1}"
            + (" (padding)" if b == 1 else " (bez paddingu)"),
            xlabel="Pozycja klucza (0–4)",
            ylabel="Pozycja zapytania (0–4)",
        )
        ax.set_xticks(range(5))
        ax.set_yticks(range(5))
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.72, label="Waga atencji")
plt.show()
print("Norma wyjść dla paddingowych zapytań:", mha_output[1, 3:].norm(dim=-1).tolist())
print("✓ wiele głów: dwie głowy, przykład z paddingiem i bez paddingu")

**Pytania.**

1. Jakie są kształty Q, K i V **jednej głowy po projekcji**, macierzy `head_weights`, połączonych wyników głów i wyjścia warstwy dla `B=2`, `S=5`, `d_model=8`, `n_heads=2`?
2. Czy dwie różne mapy losowych głów mówią cokolwiek o jakości tłumaczenia? Co można, a czego nie można z nich odczytać?
3. Po co po połączeniu głów stosujemy projekcję $W^O$?
4. Co dzieje się z wyjściami zapytań na pozycjach paddingu? Jak uwzględnić to przy liczeniu straty?

**Odpowiedzi:**

1. ...
2. ...
3. ...
4. ...

### 8. Skalowanie ocen i entropia wag

Dlaczego w scaled dot-product attention dzielimy przez $\sqrt{d_k}$? Jeśli składowe Q i K są niezależne, mają średnią zero i wariancję jeden, to:

$$q^\top k=\sum_{i=1}^{d_k}q_i k_i.$$

Przy niezależności składników $q_i k_i$ także między kolejnymi pozycjami `i`, wariancja sumy rośnie jak $d_k$. Duże wartości bezwzględne ocen mogą prowadzić do bardzo skupionego softmaxu. Dzielenie przez $\sqrt{d_k}$ utrzymuje w tym uproszczonym modelu wariancję ocen blisko `1`.

**Zadanie:** 

wykaż, że $\operatorname{Var}(q_i k_i)=1$, a następnie $\operatorname{Var}(q^\top k)=d_k$. Wystarczy wykorzystać niezależność, zerowe średnie oraz wzór na wariancję sumy.

*Twój krótki rachunek:* …

Zobaczmy teraz, jak bardzo rozkład wag jest skupiony. Przyda się entropia:

$$H(\alpha)=-\sum_j\alpha_j\ln\alpha_j.$$

**Przykład.** 

Dla rozkładu `[0.5, 0.5]` entropia wynosi $-2\cdot 0.5\ln 0.5=\ln 2$. Dla `[1, 0]` jest równa `0` (przyjmujemy $0\ln 0=0$). Entropia nie mówi, czy model przewiduje poprawnie — mówi tylko o stopniu skupienia wag.

**Zadanie:** 

uzupełnij `row_entropies`, aby obliczać entropię każdego wiersza tensora `[B,T,S]`. Zadbaj o poprawną obsługę wag równych zero.

In [ ]:
def row_entropies(weights: torch.Tensor) -> torch.Tensor:
    """Entropia każdego wiersza wag; [B,T,S] -> [B,T]."""
    # TODO: użyj H(p) = -sum_j p_j log(p_j), zabezpiecz log(0).
    raise NotImplementedError("entropia: row_entropies")

In [ ]:
uniform = torch.full((1, 2, 4), 1 / 4)
onehot = torch.tensor([[[1.0, 0.0, 0.0, 0.0]]])
mixed = torch.tensor([[[0.5, 0.5, 0.0, 0.0]]])
assert row_entropies(uniform).shape == (1, 2)
assert torch.allclose(
    row_entropies(uniform), torch.full((1, 2), math.log(4)), atol=1e-6
)
assert torch.allclose(row_entropies(onehot), torch.zeros(1, 1), atol=1e-6)
assert torch.allclose(row_entropies(mixed), torch.tensor([[math.log(2)]]), atol=1e-6)
assert torch.isfinite(row_entropies(torch.cat([onehot, mixed], dim=1))).all()
print("✓ entropia: entropia jednostajna, one-hot i z zerami")

Po uzupełnieniu funkcji `row_entropies` uruchom kod poniżej. Porównamy entropię wag wyznaczanych z surowego iloczynu QK oraz z iloczynu podzielonego przez $\sqrt{d_k}$ dla kilku wymiarów.

To tylko symulacja na losowych wektorach. Zwracaj uwagę na ogólny trend, a nie na ścisłą monotoniczność każdej liczby. Linia $\ln S$ oznacza maksymalną entropię przy rozkładzie jednostajnym na `S` pozycjach.

In [ ]:
torch.manual_seed(2026)
dims = [4, 16, 64, 256]
mean_raw, mean_scaled = [], []
for d_k in dims:
    Qexp = torch.randn(8, 5, d_k)
    Kexp = torch.randn(8, 12, d_k)
    raw = Qexp @ Kexp.transpose(-2, -1)
    raw_weights = torch.softmax(raw, dim=-1)
    scaled_weights = torch.softmax(raw / math.sqrt(d_k), dim=-1)
    mean_raw.append(row_entropies(raw_weights).mean().item())
    mean_scaled.append(row_entropies(scaled_weights).mean().item())
    print(
        f"d_k={d_k:>3}: bez skalowania H={mean_raw[-1]:.3f}, "
        f"ze skalowaniem H={mean_scaled[-1]:.3f}"
    )

xpos = np.arange(len(dims))
fig, ax = plt.subplots(figsize=(7.5, 3.8), layout="constrained")
ax.bar(xpos - 0.18, mean_raw, width=0.36, label="Bez skalowania")
ax.bar(xpos + 0.18, mean_scaled, width=0.36, label="Ze skalowaniem")
ax.axhline(math.log(12), linestyle="--", color="gray", label="ln(12): jednostajnie")
ax.set(
    xticks=xpos,
    xticklabels=[str(d) for d in dims],
    xlabel="d_k",
    ylabel="Średnia entropia H",
)
ax.legend()
plt.show()

**Pytania**

1. Jak zmienia się średnia entropia bez skalowania, gdy rośnie `d_k`? Jak wiąże się to z wariancją ocen?
2. Czy skalowanie wymusza jednostajny rozkład? Jeśli nie, to jaki efekt naprawdę daje?
3. Dlaczego wysoka entropia nie oznacza automatycznie lepszej predykcji? Czego brakuje do wyciągania wniosków o wytrenowanym modelu?

**Odpowiedzi:**

1. ...
2. ...
3. ...

### 9. Pamięć KV cache i liczba głów K/V

W autoregresyjnym generowaniu model może zapamiętywać klucze i wartości dla wcześniejszych tokenów. Dzięki temu nie musi obliczać ich od nowa przy każdym kolejnym kroku. Taka pamięć to **KV cache**. Jej wielkość zależy m.in. od liczby głów kluczy i wartości.

**Przykład.** 

Załóżmy `S=4`, `d_head=2`, jedną głowę K i V oraz `float32` (4 bajty na element). Mamy dwa tensory, K i V, z których każdy przechowuje `4 × 2` wartości. Łącznie to $2\cdot4\cdot1\cdot2\cdot4=64$ bajty. W prawdziwych modelach sekwencje i liczba warstw są oczywiście znacznie większe.

In [ ]:
demo_cache_bytes = (
    2 * 4 * 1 * 2 * 4
)  # K i V × S × liczba głów KV × d_head × bajty/element
print("Przykładowy KV cache:", demo_cache_bytes, "bajtów")
assert demo_cache_bytes == 64

**Zadanie:** 

Mamy `8` głów zapytań, `d_model=512`, więc `d_head=64`. Długość dotychczasowej sekwencji to `S=1024`, a zapis jest w formacie `float16` (2 bajty na element). Rozważamy jedną warstwę i jeden przykład.

Porównaj trzy warianty:

- **MHA** : `8` niezależnych głów K oraz V;
- **MQA** : `1` wspólną głowę K i V;
- **GQA** : `2` grupy głów K i V.

Oblicz rozmiar KV cache każdego wariantu w **MiB** ($1\,\mathrm{MiB}=2^{20}$ bajtów) oraz oszczędność procentową względem MHA. Nie wliczaj Q, wag modelu ani innych aktywacji.

Wzór pomocniczy: `bajty_KV = 2 × S × n_kv_heads × d_head × bajty_na_element`.

**Obliczenia i wniosek:**

...

## Literatura i materiały nauki

Nie trzeba czytać wszystkich prac od deski do deski. Poniżej jest kolejność, która dobrze pasuje do tego notebooka. Najpierw warto przejrzeć rysunki i równania, potem, jeśli coś zaciekawi, wrócić do metod i eksperymentów.

### Na początek: enkoder i dekoder

- **Cho, K. i in. (2014).** [*Learning Phrase Representations using RNN Encoder–Decoder for Statistical Machine Translation*](https://aclanthology.org/D14-1179/). Wprowadzenie do architektury RNN encoder–decoder, w tym do jednostki GRU.
- **Sutskever, I., Vinyals, O., Le, Q. V. (2014).** [*Sequence to Sequence Learning with Neural Networks*](https://arxiv.org/abs/1409.3215). Klasyczny model sequence-to-sequence, w którym enkoder przekazuje dekoderowi reprezentację sekwencji.

### Podstawowe publikacje

1. **Bahdanau, D., Cho, K., Bengio, Y. (2014).** [*Neural Machine Translation by Jointly Learning to Align and Translate*](https://arxiv.org/abs/1409.0473). Skąd wziął się pomysł wyznaczania osobnego kontekstu dla każdego kroku dekodera;
2. **Luong, M.-T., Pham, H., Manning, C. D. (2015).** [*Effective Approaches to Attention-based Neural Machine Translation*](https://aclanthology.org/D15-1166/). Porównanie kilku sposobów liczenia atencji w tłumaczeniu maszynowym;
3. **Vaswani, A. i in. (2017).** [*Attention Is All You Need*](https://arxiv.org/abs/1706.03762). Q/K/V, skalowanie, maski i multi-head attention;
4. **Shazeer, N. (2019).** [*Fast Transformer Decoding: One Write-Head is All You Need*](https://arxiv.org/abs/1911.02150). Dlaczego liczba głów K/V ma znaczenie podczas generowania; 
5. **Ainslie, J. i in. (2023).** [*GQA: Training Generalized Multi-Query Transformer Models from Multi-Head Checkpoints*](https://aclanthology.org/2023.emnlp-main.298/). Rozwinięcie idei MQA;
6. **Jain, S., Wallace, B. C. (2019).** [*Attention is not Explanation*](https://aclanthology.org/N19-1357/). Dlaczego na podstawie samych wag atencji trzeba ostrożnie wnioskować o przyczynach predykcji.

### Dokumentacja PyTorch

- [PyTorch: nn.MultiheadAttention](https://docs.pytorch.org/docs/stable/generated/torch.nn.MultiheadAttention.html) — m.in. wymiary tensorów, `batch_first`, `average_attn_weights` i znaczenie `key_padding_mask`.
- [PyTorch: torch.nn.functional.scaled_dot_product_attention](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html) — przydatne do porównania własnej implementacji z gotową funkcją. **Uwaga:** konwencja masek boolowskich jest w niej inna niż `key_padding_mask` w `nn.MultiheadAttention`.